# Station Stacking v15 - KATL

Experimental notebook for `KATL`.

V15 is a three-arm weather ablation family. It reruns a fresh v11-current baseline as `v15_base`, then tests `forecast_temp_at_as_of` additions and `precip_cloud` additions in separate output folders under `data/calibration/station_stacking_v15`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
TARGET_SOURCE = "iem_hourly"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
EXPORT_MODEL_WEIGHTS = True
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v15"
ARMS = [
    {
        "arm": "base",
        "feature_version": "v15_base",
        "model_version": "station_high_regressor_v15_base_v11_current_stack",
    },
    {
        "arm": "forecast_temp_at_as_of",
        "feature_version": "v15_forecast_temp_at_as_of",
        "model_version": "station_high_regressor_v15_forecast_temp_at_as_of_stack",
    },
    {
        "arm": "precip_cloud",
        "feature_version": "v15_precip_cloud",
        "model_version": "station_high_regressor_v15_precip_cloud_stack",
    },
]

PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from scripts.run_station_stacking_v15 import write_station_comparisons
from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V15_ADDITIONAL_FEATURE_COLUMNS,
    V15_BASE_FEATURE_COLUMNS,
    V15_DROPPED_FEATURE_COLUMNS,
    V15_FORECAST_TEMP_AT_AS_OF_FEATURE_COLUMNS,
    V15_PRECIP_CLOUD_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V15 Contract

`v15_base` is a fresh v11-current baseline. The added-feature arms always keep the full v11 base, block raw provider weather fields and provider-diff sprawl, and only select their explicit aggregate allowlists after train-year coverage passes.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

arm_spec = pd.DataFrame(ARMS)
fold_spec, arm_spec


(                     fold  train_start_year  train_end_year  validation_year
 0  fold_2021_2023_to_2024              2021            2023             2024
 1  fold_2021_2024_to_2025              2021            2024             2025,
                       arm             feature_version  \
 0                    base                    v15_base   
 1  forecast_temp_at_as_of  v15_forecast_temp_at_as_of   
 2            precip_cloud            v15_precip_cloud   
 
                                        model_version  
 0  station_high_regressor_v15_base_v11_current_stack  
 1  station_high_regressor_v15_forecast_temp_at_as...  
 2      station_high_regressor_v15_precip_cloud_stack  )

In [4]:
{
    "base_columns": V15_BASE_FEATURE_COLUMNS,
    "forecast_temp_at_as_of_additions": V15_FORECAST_TEMP_AT_AS_OF_FEATURE_COLUMNS,
    "precip_cloud_additions": V15_PRECIP_CLOUD_FEATURE_COLUMNS,
    "dropped_columns": sorted(V15_DROPPED_FEATURE_COLUMNS),
}


{'base_columns': ['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1998,2021-01-01,2026-06-21
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1997,2021-01-01,2026-06-21


## Run Three Arms


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

results = {}
for arm in ARMS:
    arm_dir = OUTPUT_DIR / arm["arm"]
    config = StationStackingConfig(
        station_id=STATION_ID,
        project_root=PROJECT_ROOT,
        timing_mode=TIMING_MODE,
        providers=PROVIDERS,
        fast_mode=FAST_MODE,
        optuna_trials=OPTUNA_TRIALS,
        stack_optuna_trials=STACK_OPTUNA_TRIALS,
        optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
        stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
        optuna_metric=OPTUNA_METRIC,
        optuna_verbose=OPTUNA_VERBOSE,
        feature_version=arm["feature_version"],
        target_mode="remaining_warmup",
        target_source=TARGET_SOURCE,
        base_model_methods=("xgboost", "lightgbm", "catboost"),
        stack_enabled=True,
        hyperparameter_space="wide",
        year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
        year_split_test_train_years=(2021, 2025),
        year_split_test_year=2026,
        output_dir=arm_dir,
        climatology_normals_path=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9" / "station_rolling_10y_daily_high_normals.csv",
    )
    print(f"Running {STATION_ID} {arm['arm']}: {config.resolved_optuna_storage_uri()}")
    result = run_station_year_split_experiment(config)
    results[arm["arm"]] = result
    display(result.scoreboard.assign(arm=arm["arm"], feature_version=arm["feature_version"]))
    if EXPORT_MODEL_WEIGHTS:
        exported_weights = export_station_model_weights(
            project_root=PROJECT_ROOT,
            station_id=STATION_ID,
            artifact_dir=config.resolved_output_dir(),
            model_version=arm["model_version"],
            timing_mode=config.timing_mode,
            providers=tuple(config.providers),
            feature_version=config.effective_feature_version,
            optuna_metric=config.effective_optuna_metric,
            target_mode=config.effective_target_mode,
            target_source=config.effective_target_source,
            base_model_methods=tuple(config.effective_base_model_methods),
            stack_enabled=config.stack_enabled,
            source_pipeline="notebooks/station_stacking_v15",
        )
        print(f"Exported {arm['arm']}: {exported_weights.bundle_path}")

write_station_comparisons(OUTPUT_DIR, STATION_ID, [arm["arm"] for arm in ARMS])


Running KATL base: sqlite:///D:/dev/weather-research/data/calibration/station_stacking_v15/base/KATL_optuna.sqlite3


[I 2026-06-29 15:32:16,796] A new study created in RDB with name: KATL_v15_base_remaining_warmup_base_xgboost_mae_f
[I 2026-06-29 15:32:27,133] Trial 0 finished with value: 11.672274447074303 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 11.672274447074303.
[I 2026-06-29 15:32:28,000] Trial 1 finished with value: 11.672274447074303 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 with value: 11.672274447074303.
[I 2026-06-29 15:32:44,957]

,period,method,count,mae_f,rmse_f,arm,feature_version
0,validation_2024_2025,xgboost,728,1.671761,2.384807,base,v15_base
1,validation_2024_2025,lightgbm,728,1.672786,2.417305,base,v15_base
2,validation_2024_2025,catboost,728,1.682298,2.423920,base,v15_base
3,validation_2024_2025,provider_mean,728,2.832206,4.006255,base,v15_base
4,validation_2024_2025,provider_median,728,2.782240,3.911262,base,v15_base
5,validation_2024_2025,nbm_raw,728,2.792647,3.816647,base,v15_base
6,validation_2024_2025,hrrr_raw,728,3.556181,4.953880,base,v15_base
7,validation_2024_2025,gfs_raw,728,3.254687,4.459385,base,v15_base
8,test_2026,xgboost,170,1.631589,2.319565,base,v15_base
9,test_2026,lightgbm,170,1.558784,2.223175,base,v15_base


Exported base: D:\dev\weather-research\data\calibration\station_stacking_v15\base\model_weights\KATL_station_high_regressor_v15_base_v11_current_stack.joblib
Running KATL forecast_temp_at_as_of: sqlite:///D:/dev/weather-research/data/calibration/station_stacking_v15/forecast_temp_at_as_of/KATL_optuna.sqlite3


D:\dev\weather-research\src\calibration\station_stacking.py:2622: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2621: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2622: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f,arm,feature_version
0,validation_2024_2025,xgboost,728,1.660337,2.379674,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
1,validation_2024_2025,lightgbm,728,1.648960,2.375126,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
2,validation_2024_2025,catboost,728,1.664598,2.394125,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
3,validation_2024_2025,provider_mean,728,2.832206,4.006255,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
4,validation_2024_2025,provider_median,728,2.782240,3.911262,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
5,validation_2024_2025,nbm_raw,728,2.792647,3.816647,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
6,validation_2024_2025,hrrr_raw,728,3.556181,4.953880,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
7,validation_2024_2025,gfs_raw,728,3.254687,4.459385,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
8,test_2026,xgboost,170,1.577466,2.197311,forecast_temp_at_as_of,v15_forecast_temp_at_as_of
9,test_2026,lightgbm,170,1.569194,2.208089,forecast_temp_at_as_of,v15_forecast_temp_at_as_of


Exported forecast_temp_at_as_of: D:\dev\weather-research\data\calibration\station_stacking_v15\forecast_temp_at_as_of\model_weights\KATL_station_high_regressor_v15_forecast_temp_at_as_of_stack.joblib
Running KATL precip_cloud: sqlite:///D:/dev/weather-research/data/calibration/station_stacking_v15/precip_cloud/KATL_optuna.sqlite3


D:\dev\weather-research\src\calibration\station_stacking.py:2622: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2621: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2622: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f,arm,feature_version
0,validation_2024_2025,xgboost,728,1.671957,2.391998,precip_cloud,v15_precip_cloud
1,validation_2024_2025,lightgbm,728,1.663390,2.383674,precip_cloud,v15_precip_cloud
2,validation_2024_2025,catboost,728,1.700972,2.420937,precip_cloud,v15_precip_cloud
3,validation_2024_2025,provider_mean,728,2.832206,4.006255,precip_cloud,v15_precip_cloud
4,validation_2024_2025,provider_median,728,2.782240,3.911262,precip_cloud,v15_precip_cloud
5,validation_2024_2025,nbm_raw,728,2.792647,3.816647,precip_cloud,v15_precip_cloud
6,validation_2024_2025,hrrr_raw,728,3.556181,4.953880,precip_cloud,v15_precip_cloud
7,validation_2024_2025,gfs_raw,728,3.254687,4.459385,precip_cloud,v15_precip_cloud
8,test_2026,xgboost,170,1.570912,2.202376,precip_cloud,v15_precip_cloud
9,test_2026,lightgbm,170,1.542322,2.188441,precip_cloud,v15_precip_cloud


Exported precip_cloud: D:\dev\weather-research\data\calibration\station_stacking_v15\precip_cloud\model_weights\KATL_station_high_regressor_v15_precip_cloud_stack.joblib
Wrote KATL v15 comparisons: D:\dev\weather-research\data\calibration\station_stacking_v15\KATL_v15_arm_test_metrics.csv and D:\dev\weather-research\data\calibration\station_stacking_v15\KATL_v15_arm_common_date_comparison.csv


## Arm Scoreboards


In [7]:
scoreboards = pd.concat(
    [result.scoreboard.assign(arm=arm) for arm, result in results.items()],
    ignore_index=True,
) if results else pd.DataFrame()

scoreboards.sort_values(["period", "mae_f", "arm", "method"])


,period,method,count,mae_f,rmse_f,arm
28,test_2026,ridge_stack,170,1.483075,2.125702,forecast_temp_at_as_of
27,test_2026,catboost,170,1.487267,2.134523,forecast_temp_at_as_of
45,test_2026,ridge_stack,170,1.526176,2.154990,precip_cloud
43,test_2026,lightgbm,170,1.542322,2.188441,precip_cloud
9,test_2026,lightgbm,170,1.558784,2.223175,base
11,test_2026,ridge_stack,170,1.565049,2.212661,base
26,test_2026,lightgbm,170,1.569194,2.208089,forecast_temp_at_as_of
42,test_2026,xgboost,170,1.570912,2.202376,precip_cloud
25,test_2026,xgboost,170,1.577466,2.197311,forecast_temp_at_as_of
10,test_2026,catboost,170,1.582195,2.211646,base


## Common-Date Comparison Versus Base


In [8]:
metrics_path = OUTPUT_DIR / f"{STATION_ID}_v15_arm_test_metrics.csv"
comparison_path = OUTPUT_DIR / f"{STATION_ID}_v15_arm_common_date_comparison.csv"

arm_test_metrics = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
common_date_comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()

common_date_comparison.sort_values(["method", "delta_mae_f", "comparison_arm"]) if not common_date_comparison.empty else common_date_comparison


,station_id,baseline_arm,comparison_arm,feature_version,model_version,method,common_date_count,base_mae_f,arm_mae_f,delta_mae_f,base_rmse_f,arm_rmse_f,delta_rmse_f,arm_better_days,base_better_days,tied_days,actual_mismatch_count,first_common_date,last_common_date
0,KATL,base,forecast_temp_at_as_of,v15_forecast_temp_at_as_of,station_high_regressor_v15_forecast_temp_at_as...,catboost,170,1.582195,1.487267,-0.094928,2.211646,2.134523,-0.077124,95,75,0,0,2026-01-01,2026-06-21
9,KATL,base,precip_cloud,v15_precip_cloud,station_high_regressor_v15_precip_cloud_stack,catboost,170,1.582195,1.623187,0.040992,2.211646,2.292603,0.080957,87,83,0,0,2026-01-01,2026-06-21
1,KATL,base,forecast_temp_at_as_of,v15_forecast_temp_at_as_of,station_high_regressor_v15_forecast_temp_at_as...,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
10,KATL,base,precip_cloud,v15_precip_cloud,station_high_regressor_v15_precip_cloud_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
2,KATL,base,forecast_temp_at_as_of,v15_forecast_temp_at_as_of,station_high_regressor_v15_forecast_temp_at_as...,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
11,KATL,base,precip_cloud,v15_precip_cloud,station_high_regressor_v15_precip_cloud_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
12,KATL,base,precip_cloud,v15_precip_cloud,station_high_regressor_v15_precip_cloud_stack,lightgbm,170,1.558784,1.542322,-0.016462,2.223175,2.188441,-0.034734,92,78,0,0,2026-01-01,2026-06-21
3,KATL,base,forecast_temp_at_as_of,v15_forecast_temp_at_as_of,station_high_regressor_v15_forecast_temp_at_as...,lightgbm,170,1.558784,1.569194,0.010410,2.223175,2.208089,-0.015086,85,85,0,0,2026-01-01,2026-06-21
4,KATL,base,forecast_temp_at_as_of,v15_forecast_temp_at_as_of,station_high_regressor_v15_forecast_temp_at_as...,nbm_raw,170,2.614555,2.614555,0.000000,3.750025,3.750025,0.000000,0,0,170,0,2026-01-01,2026-06-21
13,KATL,base,precip_cloud,v15_precip_cloud,station_high_regressor_v15_precip_cloud_stack,nbm_raw,170,2.614555,2.614555,0.000000,3.750025,3.750025,0.000000,0,0,170,0,2026-01-01,2026-06-21


## Added Feature Selection Audit


In [9]:
selected_rows = []
for arm, result in results.items():
    selected = set(result.feature_columns["feature"])
    selected_rows.append(
        {
            "arm": arm,
            "selected_feature_count": len(selected),
            "v15_added_features_selected": sorted(selected & set(V15_ADDITIONAL_FEATURE_COLUMNS)),
            "forecast_temp_features_selected": sorted(selected & set(V15_FORECAST_TEMP_AT_AS_OF_FEATURE_COLUMNS)),
            "precip_cloud_features_selected": sorted(selected & set(V15_PRECIP_CLOUD_FEATURE_COLUMNS)),
        }
    )

pd.DataFrame(selected_rows)


,arm,selected_feature_count,v15_added_features_selected,forecast_temp_features_selected,precip_cloud_features_selected
0,base,273,[],[],[]
1,forecast_temp_at_as_of,221,"[v13_forecast_temp_at_as_of_mean_f, v13_foreca...","[v13_forecast_temp_at_as_of_mean_f, v13_foreca...",[]
2,precip_cloud,221,"[v13_cloud_cover_max_pct, v13_cloud_cover_mean...",[],"[v13_cloud_cover_max_pct, v13_cloud_cover_mean..."


## Added Feature Coverage


In [10]:
coverage_frames = []
for arm, result in results.items():
    available = [column for column in V15_ADDITIONAL_FEATURE_COLUMNS if column in result.features]
    if not available:
        continue
    coverage = (
        result.features[available]
        .notna()
        .mean()
        .mul(100)
        .rename("coverage_pct")
        .reset_index()
        .rename(columns={"index": "feature"})
    )
    coverage["arm"] = arm
    coverage_frames.append(coverage)

added_feature_coverage = pd.concat(coverage_frames, ignore_index=True) if coverage_frames else pd.DataFrame()
added_feature_coverage.sort_values(["arm", "coverage_pct"], ascending=[True, False])


,feature,coverage_pct,arm
0,v13_forecast_temp_at_as_of_mean_f,100.000000,forecast_temp_at_as_of
1,v13_forecast_temp_at_as_of_minus_observed_mean_f,100.000000,forecast_temp_at_as_of
2,v13_forecast_temp_at_as_of_spread_f,100.000000,forecast_temp_at_as_of
3,v13_forecast_temp_bias_remaining_warmup_intera...,100.000000,forecast_temp_at_as_of
4,v13_cloud_cover_mean_pct,66.016016,forecast_temp_at_as_of
5,v13_cloud_cover_max_pct,66.016016,forecast_temp_at_as_of
6,v13_cloud_cover_remaining_warmup_interaction,66.016016,forecast_temp_at_as_of
7,v13_precip_cloud_remaining_warmup_interaction,65.915916,forecast_temp_at_as_of
8,v13_forecast_temp_at_as_of_mean_f,100.000000,precip_cloud
9,v13_forecast_temp_at_as_of_minus_observed_mean_f,100.000000,precip_cloud


## Raw Weather Sprawl Check


In [11]:
raw_weather_tokens = (
    "cloud",
    "ceiling",
    "dewpoint",
    "forecast_temp_at_as_of",
    "humidity",
    "precip",
    "pressure",
    "shortwave",
    "visibility",
    "wind_",
)
raw_weather_selected = []
for arm, result in results.items():
    selected = result.feature_columns["feature"].astype(str)
    raw_weather_selected.extend(
        {
            "arm": arm,
            "feature": feature,
        }
        for feature in selected
        if feature.startswith(("gfs_", "hrrr_", "nbm_"))
        and any(token in feature for token in raw_weather_tokens)
    )

pd.DataFrame(raw_weather_selected)


,arm,feature
0,base,gfs_precip_amount
1,base,gfs_forecast_precip_total_mm
2,base,gfs_forecast_precip_max_1h_mm
3,base,gfs_forecast_precip_hours_count
4,base,gfs_forecast_has_precip
5,base,gfs_forecast_precip_intensity_code
6,base,gfs_wind_speed_mean
7,base,gfs_wind_speed_max
8,base,gfs_wind_direction_mean
9,base,gfs_wind_gust_max


## Rounded Within 1F Accuracy


In [12]:
rounded_frames = []
for arm, result in results.items():
    preds = pd.concat(
        [
            result.validation_predictions.assign(period="validation_2024_2025"),
            result.test_predictions.assign(period="oof_2026"),
        ],
        ignore_index=True,
    )
    predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
    preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
    preds["within_1f_after_round"] = (
        pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
    ).abs().le(1)
    rounded = (
        preds
        .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
        .groupby(["period", "method"], as_index=False)
        .agg(
            count=("within_1f_after_round", "size"),
            within_1f_count=("within_1f_after_round", "sum"),
            within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
        )
    )
    rounded["arm"] = arm
    rounded_frames.append(rounded)

within_1f_accuracy_by_period = pd.concat(rounded_frames, ignore_index=True) if rounded_frames else pd.DataFrame()
within_1f_accuracy_by_period.sort_values(["period", "method", "within_1f_accuracy_pct"], ascending=[True, True, False])


,period,method,count,within_1f_count,within_1f_accuracy_pct,arm
17,oof_2026,catboost,170,111,65.294118,forecast_temp_at_as_of
34,oof_2026,catboost,170,102,60.000000,precip_cloud
0,oof_2026,catboost,170,100,58.823529,base
1,oof_2026,gfs_raw,170,47,27.647059,base
18,oof_2026,gfs_raw,170,47,27.647059,forecast_temp_at_as_of
35,oof_2026,gfs_raw,170,47,27.647059,precip_cloud
2,oof_2026,hrrr_raw,170,56,32.941176,base
19,oof_2026,hrrr_raw,170,56,32.941176,forecast_temp_at_as_of
36,oof_2026,hrrr_raw,170,56,32.941176,precip_cloud
37,oof_2026,lightgbm,170,102,60.000000,precip_cloud


## Bracket Metrics


In [13]:
bracket_frames = []
for arm, result in results.items():
    bracket = result.bracket_metrics.copy()
    bracket["arm"] = arm
    bracket_frames.append(bracket)

pd.concat(bracket_frames, ignore_index=True) if bracket_frames else pd.DataFrame()


,method,count,mae_f,rmse_f,bracket_accuracy_pct,arm
0,xgboost,170,1.631589,2.319565,40.588235,base
1,lightgbm,170,1.558784,2.223175,41.764706,base
2,catboost,170,1.582195,2.211646,42.941176,base
3,ridge_stack,170,1.565049,2.212661,40.588235,base
4,provider_mean,170,2.786086,4.048184,25.294118,base
5,provider_median,170,2.723802,3.926838,27.647059,base
6,nbm_raw,170,2.614555,3.750025,28.823529,base
7,hrrr_raw,170,3.216669,4.676677,24.705882,base
8,gfs_raw,170,3.328938,4.532080,20.588235,base
9,xgboost,170,1.577466,2.197311,37.647059,forecast_temp_at_as_of
